# 🔀 Chain Migrations — legacy chains and their LCEL equivalents

## Learning Objectives
In this notebook, you will learn:
1. **`LLMChain` → LCEL** - the same behaviour as `prompt | llm | parser`
2. **`RetrievalQA` → LCEL** - retrieval composed rather than packaged
3. **Where the legacy classes live now** - `langchain-classic`, and what that package is for
4. **Two deprecations, not one** - the retired class *and* the retired `chain(...)` call style

## Prerequisites
- `langchain >= 1.4.0`, `langchain-core >= 1.6.1`, `langchain-classic >= 1.0.8`,
  plus `langchain-openai` and `langchain-chroma` (the floors in this repo's `pyproject.toml`)
- `OPENAI_API_KEY` in your project `.env`
- Notebook `3.1_LCEL_Introduction`

> Source: <https://www.udemy.com/course/langchain-in-action-develop-llm-powered-applications/>

In [ ]:
# ============================================================================
# SETUP: load credentials from .env
# ============================================================================
from dotenv import load_dotenv

load_dotenv()

print("✅ Setup complete!")

### 🕰️ Legacy Chains

> **LangChain 1.x**: `LLMChain` is a *retired* API. It no longer ships in the
> `langchain` package — it lives in the compatibility package
> `langchain-classic`, which is why the import below reads
> `from langchain_classic.chains.llm import LLMChain`.
>
> Running the next cell emits **two** deprecation warnings, not one:
> `LLMChain` itself is deprecated (removal 2.0.0), *and* calling a chain
> directly — `chain({...})` — was replaced by `chain.invoke({...})` back in
> 0.1.0. Both styles appear below because this is the "before" half.
>
> It still runs, and it is shown deliberately so you can compare it against the
> LCEL version in the next section. **Do not reach for it in new code.**

In [ ]:
# ============================================================================
# LEGACY 0.X: LLMChain
# ============================================================================
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [("user", "Tell me a {adjective} joke")],
)

legacy_chain = LLMChain(llm=ChatOpenAI(model="gpt-4o-mini"), prompt=prompt)

legacy_result = legacy_chain({"adjective": "funny"})
legacy_result

### ✨ LCEL — the 1.x way

The same behaviour, composed with the pipe operator instead of a chain class.
LCEL is **not** deprecated: `prompt | llm | parser` is the current, supported
form, and it streams and batches without extra work.

In [ ]:
# ============================================================================
# MODERN 1.X: the same chain as an LCEL pipe
# ============================================================================
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
# llm = ChatGroq(model="openai/gpt-oss-120b")
# llm = ChatAnthropic(model="claude-sonnet-4-5")
print(f"🤖 Model loaded: {llm.model_name}")

prompt = ChatPromptTemplate.from_messages(
    [("user", "Tell me a {adjective} joke")],
)

chain = prompt | llm | StrOutputParser()

chain.invoke({"adjective": "funny"})

### 🕰️ Legacy RAG

> **LangChain 1.x**: `RetrievalQA` is retired and imports from
> `langchain-classic`. Running the next cell emits **at least two** deprecation
> warnings — one for the class, one for the retired call style
> (`chain({...})` was replaced by `chain.invoke({...})` in 0.1.0). You will usually see more: `RetrievalQA` builds an `LLMChain` and a
> `StuffDocumentsChain` internally — both themselves deprecated — and calls
> `.run()` on the latter, which is deprecated too. Count on *at least* two,
> not exactly two.
>
> **The prompt hub is worse than retired — it is currently broken for public
> prompts.** `langchain_classic.hub.pull` is deprecated (since 1.0.6, removal
> 2.0.0) *and* delegates to `langsmith` without opting in to a security gate
> added on that side, so `hub.pull("rlm/rag-prompt")` now raises:
>
> ```
> ValueError: Pulling a public prompt by owner/name is disabled by default
> because prompts may contain untrusted serialized LangChain objects.
> ```
>
> Both cells below therefore **inline** the prompt instead. The LCEL section
> explains the alternatives.

> Kept here as the "before" half of the comparison — the LCEL rewrite follows.

In [ ]:
# ============================================================================
# SETUP: a tiny in-memory vector store to retrieve from
# ============================================================================
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma


embedding_function = OpenAIEmbeddings()

docs = [
    Document(
        page_content="the dog loves to eat pizza", metadata={"source": "animal.txt"}
    ),
    Document(
        page_content="the cat loves to eat lasagna", metadata={"source": "animal.txt"}
    ),
]


db = Chroma.from_documents(docs, embedding_function)
retriever = db.as_retriever()

print(f"✅ Vector store ready: {len(docs)} document(s) indexed")

In [ ]:
# ============================================================================
# LEGACY 0.X: RetrievalQA
# ============================================================================
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate

# Original line, kept for reference -- it raises ValueError on this repo's
# pinned langsmith (see the markdown above):
#     from langchain_classic import hub
#     prompt = hub.pull("rlm/rag-prompt")
# The published "rlm/rag-prompt", inlined. See the markdown above for why
# this is no longer pulled from the hub.
prompt = ChatPromptTemplate.from_messages([
    ("human",
     "You are an assistant for question-answering tasks. Use the following "
     "pieces of retrieved context to answer the question. If you don't know "
     "the answer, just say that you don't know. Use three sentences maximum "
     "and keep the answer concise.\n"
     "Question: {question}\n"
     "Context: {context}\n"
     "Answer:"),
])

qa_chain = RetrievalQA.from_llm(llm, retriever=retriever, prompt=prompt)

# Deliberately the RETIRED call style -- `chain(...)` rather than
# `chain.invoke(...)`. Running this emits TWO deprecation warnings: one for
# RetrievalQA, one for calling a chain directly. That pairing is the point of
# this "before" cell; the LCEL section below shows the current form.
qa_chain("What does the cat like to eat?")

### ✨ LCEL — the 1.x way

The retrieval equivalent, composed rather than packaged. The retriever is just
another runnable in the pipe.

> **On the prompt: why it is inlined rather than pulled.**
> `langchain_classic.hub.pull` carries `@deprecated(since="1.0.6",
> removal="2.0.0")` and its body is a two-line delegation to the LangSmith SDK.
> On this repo's pinned `langsmith==0.12.1` that delegation omits a required
> opt-in, so pulling any `owner/name` prompt raises `ValueError` — the hub
> route does not work at all here.
>
> Your three options, in order of preference:
>
> ```python
> # 1. Inline it (what this cell does). No network, no trust decision.
> prompt = ChatPromptTemplate.from_messages([...])
>
> # 2. LangSmith SDK, explicitly accepting the risk. `langsmith` is already
> #    a pinned dependency of this repo.
> from langsmith import Client
> prompt = Client().pull_prompt(
>     "rlm/rag-prompt", dangerously_pull_public_prompt=True
> )
>
> # 3. hub.pull(...) -- deprecated AND currently raises. Do not use.
> ```
>
> The gate exists because **hub manifests are untrusted input**: they are
> executable serialized LangChain objects fetched from a shared registry.
> `hub.pull`'s own docstring carries the same warning.

In [ ]:
# ============================================================================
# MODERN 1.X: retrieval composed as LCEL
# ============================================================================
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# The published "rlm/rag-prompt", inlined. See the markdown above for why
# this is no longer pulled from the hub.
prompt = ChatPromptTemplate.from_messages([
    ("human",
     "You are an assistant for question-answering tasks. Use the following "
     "pieces of retrieved context to answer the question. If you don't know "
     "the answer, just say that you don't know. Use three sentences maximum "
     "and keep the answer concise.\n"
     "Question: {question}\n"
     "Context: {context}\n"
     "Answer:"),
])


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

qa_chain.invoke("What does the cat like to eat?")

---
## 📝 Summary

### 1. Legacy chains and their replacements
- **Key point**: `LLMChain(llm=..., prompt=...)` becomes `prompt | llm | StrOutputParser()`
- **Key point**: `RetrievalQA` becomes an LCEL pipe with the retriever as just another runnable

### 2. Where the old classes went
- **Key point**: both live in `langchain-classic` — installable and still runnable, with removal set for 2.0.0
- **Key point**: the prompt hub went further: `hub.pull` is a deprecated shim and currently raises for public prompts, so this notebook inlines the prompt instead

### 3. Two deprecations per legacy cell
- **Key point**: the class is retired *and* `chain(...)` was replaced by `chain.invoke(...)` in 0.1.0

### Next Steps
- `3.6_Chain_Migration_Advanced.ipynb` — the same treatment for `ConversationalRetrievalChain`
- `8.2_Doc_Chains_to_LCEL_LangChain_v1.ipynb` — document-combining chains (stuff / map-reduce / refine)